In [1]:
from configs import avail_models, get_avail_splits
from models import norm_vals
from video_dataset import get_wlasl_info, get_data_set, DataSetInfo, VideoDataset
import utils
import configs
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import video_transforms as vt
import torchvision.transforms.v2 as v2

In [2]:
configs.set_seed()

## Setup

### Model

Available models:

In [3]:
av_models = avail_models()
for idx, m in enumerate(av_models):
    print(f"{idx}: {m}")

0: S3D
1: R3D_18
2: R(2+1)D_18
3: Swin3D_T
4: Swin3D_S
5: Swin3D_B
6: MViTv2_S
7: MViTv1_B


Chosen model: 

In [4]:
model_name = av_models[6]
print(model_name)

MViTv2_S


Norm values: 

In [5]:
model_info = norm_vals(model_name)
print(model_info)

{'mean': (0.45, 0.45, 0.45), 'std': (0.225, 0.225, 0.225)}


### Dataset

Available splits

In [6]:
av_splits = get_avail_splits()
for s in av_splits:
    print(s)

asl100
asl2000
asl300
asl1000


Chosen split, and set

In [7]:
split_name = av_splits[0]
print(split_name)

asl100


Dataset info

In [8]:
test_info = get_wlasl_info(split_name, set_name="test")
print(test_info)


{'root': PosixPath('../data/WLASL/WLASL2000'), 'labels': PosixPath('preprocessed/labels/asl100'), 'label_suff': 'fixed_frange_bboxes_len.json', 'set_name': 'test'}


Class list

In [9]:
class_list = configs.get_class_list()
print(len(class_list))

2000


#### Data Sets

one with centre crop and one with bounding box crop and resize


In [10]:
norm_dict = norm_vals(model_name)
print(model_info)
num_frames = 16
frame_size = 112
batch_size = 1

{'mean': (0.45, 0.45, 0.45), 'std': (0.225, 0.225, 0.225)}


## Shared final transform

In [11]:
final_transform = v2.Compose(
    [
        v2.Lambda(vt._normalize_to_float),
        v2.Normalize(mean=norm_dict["mean"], std=norm_dict["std"]),
        v2.Lambda(vt._permute_time_channel),
        
    ]
)

## Centre Crop

Transform

In [ ]:
cc_tranform = v2.Compose(
    [
        v2.CenterCrop(frame_size), final_transform
    ]
)

Dataset

In [ ]:
test_set, _, _ = VideoDataset(
    set_info=test_info,
    num_frames=num_frames,
    transforms=cc_tranform,
    
)

ValueError: too many values to unpack (expected 3)